# 11. Observed vs predicted pK$_i$  -  shared, locked axes

Self-contained: cells 1-5 are copied **verbatim** from `10_additions.ipynb`, so the
curation, featurisation, scaffold split and model are byte-identical to the thesis.
Seed 0, the same split as the ROC/enrichment figure.

**Run All.** Setup takes a few minutes (featurisation); the last cell is the new part.

Produces:
- `obs_vs_pred_distribution.png` - observed vs predicted, one shared range
- `obs_vs_pred_scatter.png` - square axes, identical x and y, true 45 degree line
- `predictions_heldout.csv` - so a kernel restart never costs you this again


In [ ]:
!pip -q install rdkit xgboost catboost scikit-learn pandas matplotlib descriptastorus

In [ ]:
import os, numpy as np, pandas as pd, warnings, time; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

LOCKED = 'data/curated_15desc.csv' if os.path.exists('data/curated_15desc.csv') else 'curated_15desc.csv'
FILES  = {'5-HT2A': 'ki_verified_5-HT2A.csv', 'Dopamine D3': 'ki_verified_Dopamine_D3.csv'}
KEY    = {'5-HT2A': '5-HT2A', 'Dopamine D3': 'D3'}
for f in [LOCKED, *FILES.values()]:
    assert os.path.exists(f), f'MISSING FILE: {f}'
print('all input files found')

In [ ]:
# ---- curation + build the modelling sets (same pipeline as the thesis) ----
sr = SaltRemover(); pep = Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
def parent(s):
    m = Chem.MolFromSmiles(str(s)) if isinstance(s, str) else None
    if m is None: return None
    p = sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms() == 0:
        fr = Chem.GetMolFrags(m, asMols=True); p = max(fr, key=lambda x: x.GetNumAtoms()) if fr else m
    return p
def landrum(g):
    v = g['pKi'].round(2); v = v[~v.duplicated(keep='first')]
    v = [x for x in v if not any(abs(x-o-3.0) < 1e-6 or abs(x-o+3.0) < 1e-6 for o in v)]
    return max(v) if v else np.nan
locked = pd.read_csv(LOCKED)
def build(name):
    raw = pd.read_csv(FILES[name]); raw['standard_value'] = pd.to_numeric(raw['standard_value'], errors='coerce')
    raw = raw[raw['standard_value'] > 0].copy(); raw['pKi'] = 9.0 - np.log10(raw['standard_value'])
    raw['mol'] = raw['canonical_smiles'].apply(parent); raw = raw[raw['mol'].notna()]
    raw = raw[raw['mol'].apply(lambda m: Descriptors.MolWt(m)) < 700]
    raw = raw[~raw['mol'].apply(lambda m: m.HasSubstructMatch(pep))]
    raw['parent_smiles'] = raw['mol'].apply(Chem.MolToSmiles)
    agg = raw.groupby('molecule_chembl_id').apply(
            lambda g: pd.Series({'pKi': landrum(g), 'parent_smiles': g['parent_smiles'].iloc[0]})
          ).reset_index().dropna(subset=['pKi'])
    ex = locked[locked.target.str.contains(KEY[name], case=False)][['molecule_chembl_id','pKi','parent_smiles']].drop_duplicates('molecule_chembl_id')
    pr = agg[~agg.molecule_chembl_id.isin(ex.molecule_chembl_id)]
    both = pd.concat([ex, pr], ignore_index=True).drop_duplicates('molecule_chembl_id').reset_index(drop=True)
    print(f'{name}: {len(ex)} clinical + {len(pr)} new = {len(both)} compounds')
    return both
sets = {n: build(n) for n in FILES}

In [ ]:
# ---- featurise: ECFP4 (2048) + DescriptaStorus RDKit2DNormalized (~200) = fused ----
def featurize(dfset):
    gen = rdNormalizedDescriptors.RDKit2DNormalized()
    mols = [Chem.MolFromSmiles(s) for s in dfset.parent_smiles]; keep = [i for i, m in enumerate(mols) if m]
    dfset = dfset.iloc[keep].reset_index(drop=True); mols = [mols[i] for i in keep]; N = len(mols)
    E = np.zeros((N, 2048), np.float32)
    for i, m in enumerate(mols):
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), E[i])
    D = np.nan_to_num(np.array([gen.process(s)[1:] for s in dfset.parent_smiles], dtype=np.float32))
    scaf = np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    return np.hstack([E, D]), dfset.pKi.values.astype(float), scaf
feat = {}
for n in sets:
    t0 = time.time(); feat[n] = featurize(sets[n]); print(f'{n} featurised: X{feat[n][0].shape}  [{time.time()-t0:.0f}s]')

In [ ]:
# ---- base models + leakage-safe scaffold split (as in the thesis) ----
def boost(): return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, random_state=0, n_jobs=-1)
def rf():    return RandomForestRegressor(n_estimators=400, random_state=0, n_jobs=-1)
def scaffold_split(scaf, N, seed, tf=0.2):
    rng = np.random.RandomState(seed); g = {}
    for i, s in enumerate(scaf): g.setdefault(s, []).append(i)
    ks = list(g); rng.shuffle(ks); nt = int(round(tf*N)); te = []
    for k in ks:
        if len(te) >= nt: break
        te += g[k]
    te = set(te); return [i for i in range(N) if i not in te], sorted(te)
print('base pipeline ready')

---
## The new part

Every panel below uses **one shared x range, one shared y range and the same bins**,
so the two targets are directly comparable and nothing is flattered by auto-scaling.
The range used is printed - reuse those numbers in any other figure.


In [ ]:
# =============================================================================
#  Observed vs predicted pKi  -  SHARED, LOCKED AXES
#  Paste as ONE new cell in 10_additions.ipynb, anywhere after cell 5.
#  Uses your own objects: feat, scaffold_split(), boost()  -  nothing is redefined.
#  Seed 0, same as the enrichment/ROC figure, so it matches the thesis.
# =============================================================================
import numpy as np, matplotlib.pyplot as plt, pandas as pd

TEAL, CORAL, INK, GREY = '#116E64', '#BE5638', '#22333B', '#60727A'
PKI_LIM = None      # None = derive one shared range; or hard-code e.g. (3.5, 12.0)
NBINS   = 34

# ---- 1. held-out predictions, per target -----------------------------------
DATA = {}
for t in feat:
    X, y, scaf = feat[t]
    tr, te = scaffold_split(scaf, len(y), 0)
    yp = boost().fit(X[tr], y[tr]).predict(X[te])
    DATA[t] = (np.asarray(y)[te], np.asarray(yp))
    print(f'{t}: n_test={len(te)}')

# ---- 2. ONE range for every panel ------------------------------------------
allv = np.concatenate([np.concatenate(v) for v in DATA.values()])
lo, hi = PKI_LIM if PKI_LIM else (np.floor(allv.min()*2)/2 - 0.5,
                                  np.ceil(allv.max()*2)/2 + 0.5)
bins  = np.linspace(lo, hi, NBINS)
ticks = np.arange(np.ceil(lo), np.floor(hi) + 1, 1.0)
print(f'\nshared axis range: ({lo:.1f}, {hi:.1f})  <- reuse in every other figure')

# ---- 3. distribution: observed vs predicted --------------------------------
fig, axs = plt.subplots(1, len(DATA), figsize=(11.5, 4.0), sharex=True, sharey=True)
axs = np.atleast_1d(axs); peak = 0
for ax, (t, (yt, yp)) in zip(axs, DATA.items()):
    h1, _, _ = ax.hist(yt, bins=bins, histtype='stepfilled', alpha=0.45,
                       color=TEAL, label='observed', density=True)
    h2, _, _ = ax.hist(yp, bins=bins, histtype='step', lw=2.2,
                       color=CORAL, label='predicted', density=True)
    peak = max(peak, h1.max(), h2.max())
    ax.axvline(yt.mean(), color=TEAL, ls=':', lw=1.6)
    ax.axvline(yp.mean(), color=CORAL, ls=':', lw=1.6)
    ax.set_title(f'{t}   (SD {yt.std():.2f} obs vs {yp.std():.2f} pred)',
                 fontsize=11, color=INK)
    ax.set_xlabel('pK$_i$', color=INK); ax.set_xlim(lo, hi); ax.set_xticks(ticks)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(colors=GREY)
    ax.legend(frameon=False, fontsize=10)
    print(f'{t}:  obs mean {yt.mean():.2f} SD {yt.std():.2f}  |  '
          f'pred mean {yp.mean():.2f} SD {yp.std():.2f}  |  '
          f'variance retained {yp.std()/yt.std():.0%}')
for ax in axs:
    ax.set_ylim(0, peak * 1.15)
axs[0].set_ylabel('density', color=INK)
fig.tight_layout(); fig.savefig('obs_vs_pred_distribution.png', dpi=300,
                                bbox_inches='tight', facecolor='white'); plt.show()

# ---- 4. scatter: x and y identical, square, true 45-degree line ------------
fig, axs = plt.subplots(1, len(DATA), figsize=(10.5, 5.0), sharex=True, sharey=True)
axs = np.atleast_1d(axs)
for ax, (t, (yt, yp)) in zip(axs, DATA.items()):
    ax.scatter(yt, yp, s=9, alpha=0.35, color=TEAL, edgecolors='none')
    ax.plot([lo, hi], [lo, hi], ls='--', lw=1.4, color=CORAL)
    r2 = 1 - ((yt - yp)**2).sum() / ((yt - yt.mean())**2).sum()
    ax.set_title(f'{t}   (R$^2$ = {r2:.2f})', fontsize=11, color=INK)
    ax.set_xlabel('observed pK$_i$', color=INK)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_aspect('equal', adjustable='box')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(colors=GREY)
axs[0].set_ylabel('predicted pK$_i$', color=INK)
fig.tight_layout(); fig.savefig('obs_vs_pred_scatter.png', dpi=300,
                                bbox_inches='tight', facecolor='white'); plt.show()

# ---- 5. keep the numbers so a kernel restart never costs you this again ----
pd.concat([pd.DataFrame({'target': t, 'y_true': v[0], 'y_pred': v[1]})
           for t, v in DATA.items()], ignore_index=True) \
  .to_csv('predictions_heldout.csv', index=False)
print('\nsaved -> obs_vs_pred_distribution.png, obs_vs_pred_scatter.png, '
      'predictions_heldout.csv')
